In [22]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/movie_recsys", exist_ok=True)

API_KEY = userdata.get("TMDB_API_KEY")
print("Key loaded:", API_KEY is not None)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Key loaded: True


In [23]:
import requests

BASE = "https://api.themoviedb.org/3"

r = requests.get(f"{BASE}/search/movie",
                 params={"api_key": API_KEY, "query": "The Dark Knight"})
print("Status:", r.status_code)
movie_id = r.json()["results"][0]["id"]

r = requests.get(f"{BASE}/movie/{movie_id}",
                 params={"api_key": API_KEY, "append_to_response": "credits"})
d = r.json()

print(list(d.keys()))
print(d["title"], "|", [g["name"] for g in d["genres"]], "|", d["vote_average"])
print(d["overview"][:150])

Status: 200
['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'credits']
The Dark Knight | ['Action', 'Thriller', 'Crime'] | 8.54
Batman raises the stakes in his war on crime. With the help of Lt. Jim Gordon and District Attorney Harvey Dent, Batman sets out to dismantle the rema


In [24]:
import json, time, hashlib

CACHE_DIR = "/content/drive/MyDrive/movie_recsys/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_json(path, params=None, retries=5):
    params = dict(params or {})
    key = hashlib.md5((path + json.dumps(params, sort_keys=True)).encode()).hexdigest()
    file = f"{CACHE_DIR}/{key}.json"

    if os.path.exists(file):                      # already fetched before
        with open(file) as f:
            return json.load(f)

    for attempt in range(retries):
        try:
            r = requests.get(f"{BASE}{path}",
                             params={**params, "api_key": API_KEY}, timeout=15)
        except requests.RequestException:         # network problem
            time.sleep(2 ** attempt)
            continue

        if r.status_code == 200:
            data = r.json()
            with open(file, "w") as f:
                json.dump(data, f)
            return data
        if r.status_code == 429 or r.status_code >= 500:   # too fast or server trouble
            time.sleep(2 ** attempt)              # wait 1, 2, 4, 8, 16 seconds
            continue
        return None                               # 404 etc: do not retry
    return None

In [25]:
movie_ids, seen = [], set()
page = 1
while len(movie_ids) < 5000 and page <= 500:
    data = get_json("/discover/movie", {
        "sort_by": "vote_count.desc",
        "language": "en-US",
        "page": page,
    })
    if data:
        for m in data["results"]:
            if m["id"] not in seen:
                seen.add(m["id"])
                movie_ids.append(m["id"])
    page += 1

movie_ids = movie_ids[:5000]
print("Movie IDs:", len(movie_ids))

Movie IDs: 5000


In [26]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def fetch_details(mid):
    return get_json(f"/movie/{mid}",
                    {"append_to_response": "credits", "language": "en-US"})

with ThreadPoolExecutor(max_workers=8) as ex:
    raw_movies = list(tqdm(ex.map(fetch_details, movie_ids), total=len(movie_ids)))

raw_movies = [m for m in raw_movies if m]
print("Fetched:", len(raw_movies))

  0%|          | 0/5000 [00:00<?, ?it/s]

Fetched: 5000


In [27]:
import sqlite3
import pandas as pd

def parse(m):
    cast = m.get("credits", {}).get("cast", [])
    crew = m.get("credits", {}).get("crew", [])
    director = next((c["name"] for c in crew if c.get("job") == "Director"), None)
    return (
        m["id"],
        m.get("title"),
        (m.get("release_date") or "")[:4],
        ", ".join(g["name"] for g in m.get("genres", [])),
        m.get("overview"),
        ", ".join(c["name"] for c in cast[:5]),
        director,
        m.get("vote_average"),
        m.get("vote_count"),
        m.get("poster_path"),
    )

DB = "/content/drive/MyDrive/movie_recsys/movies.db"
con = sqlite3.connect(DB)
con.execute("""CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY, title TEXT, year TEXT, genres TEXT, plot TEXT,
    cast_names TEXT, director TEXT, rating REAL, votes INTEGER, poster TEXT)""")
con.executemany("INSERT OR REPLACE INTO movies VALUES (?,?,?,?,?,?,?,?,?,?)",
                [parse(m) for m in raw_movies])
con.commit()

print(pd.read_sql("SELECT COUNT(*) AS movies FROM movies", con))
pd.read_sql("SELECT title, year, genres, director, rating FROM movies LIMIT 5", con)

   movies
0    5000


,title,year,genres,director,rating
0,Four Rooms,1995,Comedy,Robert Rodriguez,5.900
1,Star Wars,1977,"Adventure, Action, Science Fiction",George Lucas,8.206
2,Finding Nemo,2003,"Animation, Family, Adventure",Andrew Stanton,7.823
3,Forrest Gump,1994,"Comedy, Drama, Romance",Robert Zemeckis,8.461
4,American Beauty,1999,Drama,Sam Mendes,8.000


In [28]:
df = pd.read_sql("SELECT * FROM movies ORDER BY votes DESC", con)

def blank(s):
    return s.isna() | (s.astype(str).str.strip() == "")

for col in ["plot", "cast_names", "director", "genres", "poster", "year"]:
    print(f"{col:12s} missing: {blank(df[col]).sum()}")

plot         missing: 0
cast_names   missing: 4
director     missing: 0
genres       missing: 0
poster       missing: 1
year         missing: 0


In [29]:
before = len(df)

df = df[~blank(df["plot"]) & (df["plot"].str.len() >= 30)]
df = df.drop_duplicates(subset=["title", "year"])
for col in ["cast_names", "director", "genres"]:
    df[col] = df[col].fillna("")
df = df.reset_index(drop=True)

print(before, "->", len(df), "movies")

5000 -> 5000 movies


In [30]:
def squash(text):
    return [t.strip().replace(" ", "").lower() for t in text.split(",") if t.strip()]

def make_soup(row):
    parts = [row["plot"].lower()]
    parts += squash(row["genres"])
    parts += squash(row["cast_names"])
    parts += squash(row["director"])
    return " ".join(parts)

df["soup"] = df.apply(make_soup, axis=1)
print(df.loc[df["title"] == "The Dark Knight", "soup"].values[0][:300])

batman raises the stakes in his war on crime. with the help of lt. jim gordon and district attorney harvey dent, batman sets out to dismantle the remaining criminal organizations that plague the streets. the partnership proves to be effective, but they soon find themselves prey to a reign of chaos u


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english", min_df=2)
tfidf_matrix = tfidf.fit_transform(df["soup"])

sim = cosine_similarity(tfidf_matrix).astype("float32")
print(tfidf_matrix.shape, sim.shape)

(5000, 14513) (5000, 5000)


In [32]:
import numpy as np

In [33]:
import difflib, time

titles_lower = df["title"].str.lower().tolist()

def find_movie(query):
    q = query.strip().lower()
    if q in titles_lower:
        return titles_lower.index(q)
    match = difflib.get_close_matches(q, titles_lower, n=1, cutoff=0.6)
    return titles_lower.index(match[0]) if match else None

def recommend(title, n=5):
    idx = find_movie(title)
    if idx is None:
        return f"No close match for '{title}'"
    scores = sim[idx].copy()
    scores[idx] = -1
    top = np.argsort(scores)[::-1][:n]
    out = df.loc[top, ["title", "year", "genres", "director", "rating"]].copy()
    out["score"] = scores[top].round(3)
    return out

print(recommend("the dark knight"))
print(recommend("dark knigt"))

t = time.time(); recommend("Inception")
print("Time:", round(time.time() - t, 4), "sec")

                         title  year  \
24       The Dark Knight Rises  2012   
41               Batman Begins  2005   
476                     Batman  1989   
904             Batman Forever  1995   
2867  Batman: The Killing Joke  2016   

                                         genres           director  rating  \
24                      Action, Crime, Thriller  Christopher Nolan   7.801   
41                         Drama, Crime, Action  Christopher Nolan   7.733   
476                      Fantasy, Action, Crime         Tim Burton   7.235   
904                      Action, Crime, Fantasy    Joel Schumacher   5.463   
2867  Animation, Crime, Drama, Action, Thriller            Sam Liu   6.602   

      score  
24    0.311  
41    0.217  
476   0.214  
904   0.179  
2867  0.160  
                         title  year  \
24       The Dark Knight Rises  2012   
41               Batman Begins  2005   
476                     Batman  1989   
904             Batman Forever  1995   
2867  

In [34]:
from scipy.sparse import hstack

df["genre_txt"] = df["genres"].apply(lambda s: " ".join(squash(s)))
df["cast_txt"]  = df["cast_names"].apply(lambda s: " ".join(squash(s)))
df["dir_txt"]   = df["director"].apply(lambda s: " ".join(squash(s)))

M_plot  = TfidfVectorizer(stop_words="english", min_df=2).fit_transform(df["plot"].str.lower())
M_genre = TfidfVectorizer().fit_transform(df["genre_txt"])
M_cast  = TfidfVectorizer(min_df=2).fit_transform(df["cast_txt"])
M_dir   = TfidfVectorizer(min_df=2).fit_transform(df["dir_txt"])

def build_sim(w_plot, w_genre, w_cast, w_dir):
    M = hstack([M_plot * w_plot, M_genre * w_genre,
                M_cast * w_cast, M_dir * w_dir]).tocsr()
    return cosine_similarity(M).astype("float32")

sim_w = build_sim(w_plot=1.0, w_genre=1.5, w_cast=0.5, w_dir=1.0)

In [35]:
def recommend(title, n=5, matrix=None):
    matrix = sim if matrix is None else matrix
    idx = find_movie(title)
    if idx is None:
        return f"No close match for '{title}'"
    scores = matrix[idx].copy()
    scores[idx] = -1
    top = np.argsort(scores)[::-1][:n]
    out = df.loc[top, ["title", "year", "genres", "director"]].copy()
    out["score"] = scores[top].round(3)
    return out

In [36]:
sim_strong = build_sim(w_plot=0.7, w_genre=2.0, w_cast=0.5, w_dir=1.5)

for movie in ["The Dark Knight", "Toy Story", "The Godfather", "Inception"]:
    print("=====", movie)
    print("OLD:\n", recommend(movie, matrix=sim)[["title", "year"]].to_string(index=False))
    print("NEW:\n", recommend(movie, matrix=sim_w)[["title", "year"]].to_string(index=False))
    print("STRONG:\n", recommend(movie, matrix=sim_strong)[["title", "year"]].to_string(index=False))

===== The Dark Knight
OLD:
                    title year
   The Dark Knight Rises 2012
           Batman Begins 2005
                  Batman 1989
          Batman Forever 1995
Batman: The Killing Joke 2016
NEW:
                  title year
 The Dark Knight Rises 2012
         Batman Begins 2005
              Insomnia 2002
  The Man from Nowhere 2010
District 13: Ultimatum 2009
STRONG:
                 title year
The Dark Knight Rises 2012
        Batman Begins 2005
             Insomnia 2002
            Inception 2010
 The Man from Nowhere 2010
===== Toy Story
OLD:
                  title year
           Toy Story 3 2010
           Toy Story 2 1999
           Toy Story 4 2019
The 40 Year-Old Virgin 2005
           Toy Story 5 2026
NEW:
                      title year
               Toy Story 2 1999
              A Bug's Life 1998
                      Cars 2006
                    Cars 2 2011
Toy Story That Time Forgot 2014
STRONG:
                      title year
              A Bu

In [37]:
final_sim = sim_w        # or sim_strong, or your own weights

In [38]:
t = time.time()
for _ in range(20):
    recommend("Inception", matrix=final_sim)
print("Average time per recommendation:", round((time.time() - t) / 20, 4), "sec")

np.save("/content/drive/MyDrive/movie_recsys/final_sim.npy", final_sim)
df.to_pickle("/content/drive/MyDrive/movie_recsys/movies_clean.pkl")

Average time per recommendation: 0.0022 sec


In [39]:
tests = ["The Dark Knight", "Toy Story", "The Godfather", "Inception", "Titanic"]

lines = ["| Movie | Top 5 recommendations |", "|---|---|"]
for t in tests:
    res = recommend(t, matrix=final_sim)
    if isinstance(res, str):
        print("Skipped:", res)
        continue
    lines.append(f"| {t} | " + ", ".join(res["title"]) + " |")

print("\n".join(lines))

| Movie | Top 5 recommendations |
|---|---|
| The Dark Knight | The Dark Knight Rises, Batman Begins, Insomnia, The Man from Nowhere, District 13: Ultimatum |
| Toy Story | Toy Story 2, A Bug's Life, Cars, Cars 2, Toy Story That Time Forgot |
| The Godfather | The Godfather Part II, The Outsiders, The Godfather Part III, Extremely Wicked, Shockingly Evil and Vile, The Unforgivable |
| Inception | The Dark Knight Rises, Tenet, Batman Begins, Demolition Man, Code 8 |
| Titanic | Desire, The Room, Below Her Mouth, Life in a Year, Notes on a Scandal |


In [ ]:
!pip install -q gradio
import gradio as gr

IMG = "https://image.tmdb.org/t/p/w342"
NO_POSTER = "https://placehold.co/342x513?text=No+poster"

def ui_recommend(title):
    idx = find_movie(title)
    if idx is None:
        return [], f"No close match for '{title}'. Try another title."

    scores = final_sim[idx].copy()
    scores[idx] = -1
    top = np.argsort(scores)[::-1][:5]

    cards = []
    for i in top:
        row = df.loc[i]
        poster = f"{IMG}{row['poster']}" if row["poster"] else NO_POSTER
        cards.append((poster, f"{row['title']} ({row['year']})"))

    picked = df.loc[idx]
    return cards, f"Movies similar to **{picked['title']} ({picked['year']})**"

with gr.Blocks(title="Movie Recommender") as demo:
    gr.Markdown("# 🎬 Movie Recommender\nType a movie name. Small typos are fine.")
    box = gr.Textbox(label="Movie title", placeholder="e.g. dark knigt")
    btn = gr.Button("Recommend")
    msg = gr.Markdown()
    gallery = gr.Gallery(label="You may also like", columns=5, height="auto")
    gr.Markdown("*This product uses the TMDB API but is not endorsed or certified by TMDB.*")

    btn.click(ui_recommend, box, [gallery, msg])
    box.submit(ui_recommend, box, [gallery, msg])

demo.launch(share=True)